In [0]:
%pip install -qqqq -U mlflow-skinny[databricks] databricks-sdk databricks-openai "psycopg[binary]>=3.1.0"
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
default_warehouse = next(
    (
        wh
        for wh in w.warehouses.list()
        if "Serverless Starter Warehouse" in wh.name and wh.enable_serverless_compute
    ),
    None,
)
default_warehouse_id = default_warehouse.id if default_warehouse else None
print(f"{default_warehouse_id=}")

In [0]:
import mlflow
import os
from databricks_openai import DatabricksOpenAI

os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = default_warehouse_id
# Enable MLflow's autologging to instrument your application with Tracing
mlflow.openai.autolog()

# Set up MLflow tracking to Databricks
mlflow.set_tracking_uri("databricks")
EXPERIMENT_NAME = "/Workspace/Shared/bo-uc-eval-traces-batch"
mlflow.set_experiment(EXPERIMENT_NAME)

# Create an OpenAI client that is connected to Databricks-hosted LLMs
client = DatabricksOpenAI()

# Select an LLM
model_name = "databricks-claude-opus-5"

In [0]:
from typing import Dict

# Example questions about Databricks code patterns (grounded in indexed repos)
GENERIC_QUESTIONS = [
    "Show me how to import DatabricksOpenAI from databricks_openai and use it with WorkspaceClient to call chat.completions.create",
    # "what are Databricks security best practices",
    # "What is Databricks used for?",
    # "How does Databricks integrate with Apache Spark?",
    # "What are the benefits of using Databricks notebooks?",
    # "Can Databricks be used for machine learning?",
    # "How does Databricks handle data security?",
]

@mlflow.trace
def answer_databricks_question(question: str) -> Dict[str, str]:
    """Generate a code-grounded answer using BM25 retrieval from Lakebase."""
    # Retrieve relevant code snippets first (RAG)
    try:
        code_context = _search_code_bm25(question, top_k=3)
    except Exception:
        code_context = "No code context available."

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a Databricks code assistant. Answer questions based ONLY on "
                    "the provided code snippets. Do not invent APIs or patterns not shown "
                    "in the code. If the code shows a specific import or usage pattern, "
                    "use exactly that pattern in your answer."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Question: {question}\n\n"
                    f"Reference code:\n{code_context}"
                ),
            },
        ],
        max_tokens=1000,
    )

    content = response.choices[0].message.content
    # Extract text from content blocks if it's a list (e.g., reasoning + text blocks)
    if isinstance(content, list):
        content = "\n".join(
            block.get("text", "") if isinstance(block, dict) and block.get("type") == "text"
            else ""
            for block in content
        ).strip()
    return {"answer": content}

# Test the application
for q in GENERIC_QUESTIONS:
    result = answer_databricks_question(q)
    print(f"Q: {q}\nA: {result['answer']}\n")

In [0]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
traces_df = mlflow.search_traces(
    locations=[experiment.experiment_id],
    order_by=["timestamp DESC"],
    max_results=1,
)
print(f"Found {len(traces_df)} traces to evaluate")

In [0]:
import psycopg
from databricks.sdk import WorkspaceClient
from mlflow.genai.scorers import scorer
from mlflow.entities import Feedback

# Lakebase connection config for the code-search project
LAKEBASE_PROJECT = "code-search"
LAKEBASE_BRANCH = "production"
LAKEBASE_ENDPOINT = "primary"
LAKEBASE_DB = "databricks_postgres"


def _search_code_bm25(query: str, top_k: int = 5) -> str:
    """Perform BM25 keyword search against the Lakebase code-search project."""
    ws = WorkspaceClient()
    endpoint_name = f"projects/{LAKEBASE_PROJECT}/branches/{LAKEBASE_BRANCH}/endpoints/{LAKEBASE_ENDPOINT}"
    endpoint = ws.postgres.get_endpoint(name=endpoint_name)
    host = endpoint.status.hosts.host
    user = ws.current_user.me().user_name
    token = ws.postgres.generate_database_credential(endpoint=endpoint_name).token

    sql = """
        SELECT c.content, f.path, r.name AS repo_name, r.default_branch,
               c.start_line, c.end_line,
               c.ts <@> to_bm25query(to_tsvector('english', %(query)s), 'ix_chunks_ts_bm25'::regclass) AS score
        FROM chunks c
        JOIN files f ON f.id = c.file_id
        JOIN repos r ON r.id = f.repo_id
        ORDER BY c.ts <@> to_bm25query(to_tsvector('english', %(query)s), 'ix_chunks_ts_bm25'::regclass)
        LIMIT %(top_k)s
    """

    with psycopg.connect(
        host=host, dbname=LAKEBASE_DB, user=user, password=token, sslmode="require"
    ) as conn:
        with conn.cursor() as cur:
            cur.execute(sql, {"query": query, "top_k": top_k})
            rows = cur.fetchall()

    if not rows:
        return "No relevant code found."

    snippets = []
    for content, path, repo_name, branch, start_line, end_line, score in rows:
        url = f"https://github.com/{repo_name}/blob/{branch}/{path}#L{start_line}-L{end_line}"
        snippets.append(f"--- {repo_name}: {path} (score: {score:.4f}) ---\n{url}\n{content}")
    return "\n\n".join(snippets)


@scorer
def code_grounded(inputs, outputs) -> Feedback:
    """Score whether the response is grounded in actual code from BM25 search."""
    question = inputs.get("query", inputs.get("question", str(inputs)))

    # Retrieve relevant code snippets via Lakebase BM25 search
    try:
        code_snippets = _search_code_bm25(question)
    except Exception as e:
        return Feedback(
            value=False,
            rationale=f"Code search unavailable: {type(e).__name__}: {e}",
        )

    if code_snippets == "No relevant code found.":
        return Feedback(
            value=False,
            rationale="No relevant code snippets found for this question.",
        )

    # Use LLM to judge if the output is consistent with the retrieved code
    judge_response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an evaluation judge. Given a question, an answer, and "
                    "reference code snippets, determine whether the answer is factually "
                    "grounded in the code snippets. Respond with ONLY 'yes' or 'no' on "
                    "the first line, followed by a brief rationale on the next line."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Question: {question}\n\n"
                    f"Answer: {outputs}\n\n"
                    f"Reference code snippets:\n{code_snippets}"
                ),
            },
        ],
        max_tokens=300,
    )

    content = judge_response.choices[0].message.content
    # Handle content as list of blocks (newer API format) or plain string
    if isinstance(content, list):
        content = "".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in content
        )
    judge_text = content.strip()
    lines = judge_text.split("\n", 1)
    verdict = lines[0].strip().lower() == "yes"
    rationale = lines[1].strip() if len(lines) > 1 else judge_text

    return Feedback(
        value=verdict,
        rationale=f"{rationale}\n\n--- Retrieved Code Snippets ---\n{code_snippets}",
    )

In [0]:
eval_results = mlflow.genai.evaluate(
    data=traces_df,
    scorers=[
        code_grounded
    ],
)

In [0]:
print("Metrics:", eval_results.metrics)
print()
print("Available tables:", list(eval_results.tables.keys()))
for name, df in eval_results.tables.items():
    print(f"\n--- {name} ---")
    print(df.columns.tolist())
    for _, row in df.iterrows():
        val = row.get('code_grounded/value', row.get('value', 'N/A'))
        rat = row.get('code_grounded/rationale', row.get('rationale', ''))
        print(f"\ncode_grounded value: {val}")
        print(f"Rationale (first 800 chars):\n{str(rat)[:800]}")